# 01 — AI Agent Without MCP

**What you'll learn**
- How an AI agent looks when the host owns its tools directly.
- Why this is fine for a tiny demo but painful at scale.
- The exact pain points MCP was designed to solve (we'll fix them in notebook 02).

## Architecture

```text
+---------------------+
| Your app (host)     |
|  - LLM prompt       |
|  - Tool definitions |
|  - Tool execution   |
+----------+----------+
           |
           v
+---------------------+
| Hardcoded Python    |
| functions           |
+----------+----------+
           |
           v
+---------------------+
| Fake CRM (in-memory)|
+---------------------+
```

Everything lives in one process. No protocol, no boundary. If Claude Desktop, Cursor, or another agent wants the same tools, they have to copy your code.

## Step 1 — Set up a fake CRM

We mock HubSpot so the notebook runs anywhere. The exact storage doesn't matter; what matters is that something has to hold state.

In [ ]:
# Fake in-memory CRM. In production this would be HubSpot, Salesforce, etc.
CONTACTS: dict = {}
TASKS: list = []
OSC_TEAM = [
    {"id": "osc_101", "name": "Ava OSC",  "last_assigned": 0},
    {"id": "osc_102", "name": "Ben OSC",  "last_assigned": 0},
    {"id": "osc_103", "name": "Cara OSC", "last_assigned": 0},
]
_assignment_counter = 0
print("Fake CRM ready. Contacts:", len(CONTACTS), "OSCs:", len(OSC_TEAM))

## Step 2 — Define the tool functions

Three plain Python functions. Notice they have no protocol, no schema, no description metadata — the host has to know about them out of band.

In [ ]:
def create_contact(name: str, email: str) -> dict:
    """Add a new contact to the CRM."""
    contact_id = f"contact_{len(CONTACTS) + 1}"
    contact = {"id": contact_id, "name": name, "email": email, "owner_id": None}
    CONTACTS[contact_id] = contact
    return contact


def assign_osc(contact_id: str) -> dict:
    """Assign the contact to the OSC with the lowest recent-assignment counter (round robin)."""
    global _assignment_counter
    if contact_id not in CONTACTS:
        raise ValueError(f"Contact not found: {contact_id}")
    chosen = min(OSC_TEAM, key=lambda osc: osc["last_assigned"])
    _assignment_counter += 1
    chosen["last_assigned"] = _assignment_counter
    CONTACTS[contact_id]["owner_id"] = chosen["id"]
    return chosen


def create_followup_task(contact_id: str, note: str) -> dict:
    """Create an open follow-up task for a contact."""
    if contact_id not in CONTACTS:
        raise ValueError(f"Contact not found: {contact_id}")
    task = {
        "id": f"task_{len(TASKS) + 1}",
        "contact_id": contact_id,
        "note": note,
        "status": "open",
    }
    TASKS.append(task)
    return task

## Step 3 — The agent's planner

In a real app the LLM looks at the user message plus tool definitions and emits a plan. We keep a deterministic mock so the notebook runs offline; if you set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` you can swap in a real call at the marked spot.

The placeholder `$last.id` means "use the `id` field from the previous tool's result".

In [ ]:
import os

def llm_plan(user_message: str) -> list[dict]:
    """Return the agent's tool-call plan.

    If OPENAI_API_KEY or ANTHROPIC_API_KEY is set, you would normally call the LLM
    here with the available tool schemas and let it choose. To keep this notebook
    100% runnable offline, we always return a deterministic mock plan. The shape
    is what a real LLM tool-use response would give you after parsing.

    Each step optionally has a "bind" name. Later steps reference earlier
    results with "$<bind>.<field>". This mirrors how real LLM tool-calling
    chains outputs across calls.
    """
    if os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"):
        # Real implementation hook — left as a comment so the example still runs
        # without a key. Sketch:
        #   from openai import OpenAI
        #   client = OpenAI()
        #   resp = client.chat.completions.create(model="gpt-4o", messages=[...], tools=[...])
        #   return parse_tool_calls(resp)
        pass

    return [
        {"tool": "create_contact",
         "args": {"name": "John Doe", "email": "john@example.com"},
         "bind": "contact"},
        {"tool": "assign_osc",
         "args": {"contact_id": "$contact.id"}},
        {"tool": "create_followup_task",
         "args": {"contact_id": "$contact.id",
                  "note": "Follow up with John Doe within 24 hours"}},
    ]


def resolve_placeholders(args: dict, bindings: dict) -> dict:
    """Replace $<bind>.<field> tokens with values from earlier tool results."""
    resolved = {}
    for k, v in args.items():
        if isinstance(v, str) and v.startswith("$"):
            ref, field = v[1:].split(".", 1)
            if ref not in bindings:
                raise ValueError(f"unknown binding: {ref}")
            resolved[k] = bindings[ref][field]
        else:
            resolved[k] = v
    return resolved

## Step 4 — The agent loop

The host:
1. Asks the planner for a list of tool calls.
2. Walks the list, resolving placeholders.
3. Calls the tool functions **directly by reference** — there is no indirection.

In [ ]:
TOOLS = {
    "create_contact": create_contact,
    "assign_osc": assign_osc,
    "create_followup_task": create_followup_task,
}

def run_agent_without_mcp(user_message: str) -> list[dict]:
    plan = llm_plan(user_message)
    results = []
    bindings: dict = {}
    for step in plan:
        fn = TOOLS[step["tool"]]
        args = resolve_placeholders(step["args"], bindings)
        result = fn(**args)
        results.append({"tool": step["tool"], "args": args, "result": result})
        if "bind" in step:
            bindings[step["bind"]] = result
    return results

## Step 5 — Run it

In [ ]:
import json

results = run_agent_without_mcp("Create a contact for John and assign an OSC")
print(json.dumps(results, indent=2))
print()
print("CONTACTS:", CONTACTS)
print("TASKS:", TASKS)

## Mini test

A real test suite would do this with pytest. We assert inline so the notebook stays self-contained.

In [ ]:
assert len(results) == 3, "expected 3 tool calls"
assert results[0]["tool"] == "create_contact"
assert results[1]["tool"] == "assign_osc"
assert results[2]["tool"] == "create_followup_task"
assert CONTACTS["contact_1"]["owner_id"].startswith("osc_")
assert TASKS[0]["status"] == "open"
print("ok")

## Why this hurts at scale

| Problem | What it looks like |
|---|---|
| **Tight coupling** | Tools live in the same module as the agent. Swap the host and you reimplement them. |
| **No discovery** | Another app can't ask "what can you do?" — it has to read your source. |
| **No reuse** | Three agents that need `create_contact` end up with three copies. |
| **No clean auth boundary** | The function is just a Python call; you can't reject an unauthorized caller. |
| **No transport** | The tools cannot move to another machine without rewriting everything. |
| **No versioning** | If you change `create_contact`, every caller breaks the same day. |

MCP is the standard that solves all of these by putting tools behind a small, well-defined server interface. That's notebook 02.

## Key takeaway

When the agent and the tools live in the same code, **the tools belong to that one app**. MCP turns tools into a reusable service that any AI app can discover and call.